In [39]:
from pathlib import Path
import random
import copy

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import (
    TensorDataset,
    DataLoader,
)

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    precision_recall_curve,
)

In [40]:
SEED = 42


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)


set_seed(SEED)


ROOT = next(
    (
        p
        for p in [
            Path.cwd(),
            *Path.cwd().parents,
        ]
        if ((p / "data").is_dir() and (p / "notebooks").is_dir())
    ),
    Path.cwd(),
)


HDA_DIR = ROOT / "data" / "processed" / "hda"

MODEL_DIR = ROOT / "models"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")

elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")

else:
    DEVICE = torch.device("cpu")


print("Device:", DEVICE)
print("HDA_DIR:", HDA_DIR)

Device: mps
HDA_DIR: /Users/thonph/Desktop/KLTN/data/processed/hda


In [41]:
# SOURCE
X_s_train = np.load(
    HDA_DIR / "X_s_train.npy",
    mmap_mode="c",
)

X_s_val = np.load(
    HDA_DIR / "X_s_val.npy",
    mmap_mode="c",
)

X_s_test = np.load(
    HDA_DIR / "X_s_test.npy",
    mmap_mode="c",
)

y_s_train = np.load(HDA_DIR / "y_s_train.npy")

y_s_val = np.load(HDA_DIR / "y_s_val.npy")

y_s_test = np.load(HDA_DIR / "y_s_test.npy")


# TARGET
X_t_train = np.load(
    HDA_DIR / "X_t_train.npy",
    mmap_mode="c",
)

X_t_test = np.load(
    HDA_DIR / "X_t_test.npy",
    mmap_mode="c",
)

# TARGET LABELS:
# evaluation only
y_t_test = np.load(HDA_DIR / "y_t_test.npy")


print("Source train:", X_s_train.shape)
print("Source val:", X_s_val.shape)
print("Source test:", X_s_test.shape)

print("Target adaptation:", X_t_train.shape)
print("Target test:", X_t_test.shape)

Source train: (1441582, 203)
Source val: (308911, 203)
Source test: (308911, 203)
Target adaptation: (2259960, 69)
Target test: (564991, 69)


In [42]:
SOURCE_DIM = X_s_train.shape[1]
TARGET_DIM = X_t_train.shape[1]

LATENT_DIM = 64


assert SOURCE_DIM == 203
assert TARGET_DIM == 69
assert LATENT_DIM == 64

assert X_s_train.dtype == np.float32
assert X_t_train.dtype == np.float32

assert np.isfinite(X_s_train).all()
assert np.isfinite(X_t_train).all()


print("SOURCE_DIM:", SOURCE_DIM)
print("TARGET_DIM:", TARGET_DIM)
print("LATENT_DIM:", LATENT_DIM)

SOURCE_DIM: 203
TARGET_DIM: 69
LATENT_DIM: 64


In [43]:
source_train_dataset = TensorDataset(
    torch.from_numpy(np.asarray(X_s_train)),
    torch.from_numpy(y_s_train.astype(np.float32)),
)


source_val_dataset = TensorDataset(
    torch.from_numpy(np.asarray(X_s_val)),
    torch.from_numpy(y_s_val.astype(np.float32)),
)


source_test_dataset = TensorDataset(
    torch.from_numpy(np.asarray(X_s_test)),
    torch.from_numpy(y_s_test.astype(np.float32)),
)


target_train_dataset = TensorDataset(
    torch.from_numpy(np.asarray(X_t_train)),
)


target_test_dataset = TensorDataset(
    torch.from_numpy(np.asarray(X_t_test)),
    torch.from_numpy(y_t_test.astype(np.float32)),
)

In [44]:
BATCH_SIZE = 512


source_train_loader = DataLoader(
    source_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=0,
)


target_train_loader = DataLoader(
    target_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=0,
)


source_val_loader = DataLoader(
    source_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=0,
)


source_test_loader = DataLoader(
    source_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=0,
)


target_test_loader = DataLoader(
    target_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=0,
)


print("Source train batches:", len(source_train_loader))

print("Target train batches:", len(target_train_loader))
# ============================================================
# FIXED LOADERS FOR BEFORE/AFTER MMD COMPARISON
# ============================================================

source_mmd_loader = DataLoader(
    source_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=True,
    num_workers=0,
)

target_mmd_loader = DataLoader(
    target_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=True,
    num_workers=0,
)

print(
    "MMD evaluation loaders ready:",
    len(source_mmd_loader),
    len(target_mmd_loader),
)

Source train batches: 2815
Target train batches: 4413
MMD evaluation loaders ready: 2815 4413


In [45]:
class SourceEncoder(nn.Module):

    def __init__(
        self,
        input_dim,
        latent_dim=64,
    ):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(
                input_dim,
                256,
            ),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(
                256,
                128,
            ),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(
                128,
                latent_dim,
            ),
        )

    def forward(self, x):
        return self.encoder(x)

In [46]:
class TargetEncoder(nn.Module):

    def __init__(
        self,
        input_dim,
        latent_dim=64,
    ):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(
                input_dim,
                256,
            ),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(
                256,
                128,
            ),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(
                128,
                latent_dim,
            ),
        )

    def forward(self, x):
        return self.encoder(x)

In [47]:
class BinaryClassifier(nn.Module):

    def __init__(
        self,
        latent_dim=64,
    ):
        super().__init__()

        self.classifier = nn.Sequential(
            nn.Linear(
                latent_dim,
                32,
            ),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(
                32,
                1,
            ),
        )

    def forward(self, z):

        return self.classifier(z).squeeze(1)

In [48]:
PRETRAINED_PATH = MODEL_DIR / "source_pretrained.pt"


checkpoint = torch.load(
    PRETRAINED_PATH,
    map_location="cpu",
    weights_only=False,
)


assert checkpoint["source_dim"] == SOURCE_DIM

assert checkpoint["latent_dim"] == LATENT_DIM


print("Pretrained best epoch:", checkpoint["best_epoch"])

Pretrained best epoch: 28


In [49]:
pretrained_source_metrics = checkpoint.get("source_test_metrics")

print("=" * 60)
print("SOURCE PRETRAINING BASELINE")
print("=" * 60)

if pretrained_source_metrics is not None:

    print("PR-AUC:", pretrained_source_metrics["pr_auc"])

    print("ROC-AUC:", pretrained_source_metrics["roc_auc"])

    print("F1:", pretrained_source_metrics["f1"])

else:

    print("source_test_metrics " "not found in checkpoint.")

SOURCE PRETRAINING BASELINE
PR-AUC: 0.9787953034411052
ROC-AUC: 0.9988618534943897
F1: 0.9071749471458774


In [50]:
source_encoder = SourceEncoder(
    input_dim=SOURCE_DIM,
    latent_dim=LATENT_DIM,
)

target_encoder = TargetEncoder(
    input_dim=TARGET_DIM,
    latent_dim=LATENT_DIM,
)

classifier = BinaryClassifier(
    latent_dim=LATENT_DIM,
)


source_encoder.load_state_dict(checkpoint["source_encoder_state_dict"])

classifier.load_state_dict(checkpoint["classifier_state_dict"])


source_encoder = source_encoder.to(DEVICE)

target_encoder = target_encoder.to(DEVICE)

classifier = classifier.to(DEVICE)


print("Models initialized.")

Models initialized.


In [51]:
xs, ys = next(iter(source_train_loader))

(xt,) = next(iter(target_train_loader))


xs = xs.to(
    DEVICE,
    dtype=torch.float32,
)

xt = xt.to(
    DEVICE,
    dtype=torch.float32,
)


with torch.no_grad():

    zs = source_encoder(xs)

    zt = target_encoder(xt)

    logits_s = classifier(zs)


print("Source input:", xs.shape)
print("Target input:", xt.shape)

print("Source latent:", zs.shape)
print("Target latent:", zt.shape)

print("Source logits:", logits_s.shape)


assert zs.shape == (
    BATCH_SIZE,
    LATENT_DIM,
)

assert zt.shape == (
    BATCH_SIZE,
    LATENT_DIM,
)

assert zs.shape[1] == zt.shape[1]

Source input: torch.Size([512, 203])
Target input: torch.Size([512, 69])
Source latent: torch.Size([512, 64])
Target latent: torch.Size([512, 64])
Source logits: torch.Size([512])


In [52]:
def pairwise_sq_dist(
    x,
    y,
):

    x_norm = x.pow(2).sum(
        dim=1,
        keepdim=True,
    )

    y_norm = (
        y.pow(2)
        .sum(
            dim=1,
            keepdim=True,
        )
        .T
    )

    dist = x_norm + y_norm - 2.0 * x @ y.T

    return dist.clamp_min(0.0)

In [53]:
@torch.no_grad()
def median_bandwidth(
    x,
    y,
):

    combined = torch.cat(
        [x, y],
        dim=0,
    )

    distances = pairwise_sq_dist(
        combined,
        combined,
    )

    n = distances.size(0)

    mask = ~torch.eye(
        n,
        dtype=torch.bool,
        device=distances.device,
    )

    values = distances[mask]

    median = torch.median(values)

    return median.clamp_min(1e-6)

In [54]:
def rbf_kernel(
    x,
    y,
    sigma2,
):

    distances = pairwise_sq_dist(
        x,
        y,
    )

    return torch.exp(-distances / (2.0 * sigma2))

In [55]:
def mmd_rbf(
    source,
    target,
):

    n_s = source.size(0)
    n_t = target.size(0)

    if n_s < 2 or n_t < 2:
        raise ValueError("MMD requires batch size >= 2.")

    sigma2 = median_bandwidth(
        source.detach(),
        target.detach(),
    )

    K_ss = rbf_kernel(
        source,
        source,
        sigma2,
    )

    K_tt = rbf_kernel(
        target,
        target,
        sigma2,
    )

    K_st = rbf_kernel(
        source,
        target,
        sigma2,
    )

    source_term = (K_ss.sum() - torch.diagonal(K_ss).sum()) / (n_s * (n_s - 1))

    target_term = (K_tt.sum() - torch.diagonal(K_tt).sum()) / (n_t * (n_t - 1))

    cross_term = K_st.mean()

    mmd2 = source_term + target_term - 2.0 * cross_term

    return mmd2

In [56]:
source_encoder.eval()
target_encoder.eval()


with torch.no_grad():

    zs = source_encoder(xs)

    zt = target_encoder(xt)

    test_mmd = mmd_rbf(
        zs,
        zt,
    )


print("Initial batch MMD²:", test_mmd.item())

assert torch.isfinite(test_mmd)

Initial batch MMD²: 0.9603609442710876


In [57]:
pos_weight_value = checkpoint.get(
    "pos_weight",
    ((y_s_train == 0).sum() / (y_s_train == 1).sum()),
)


pos_weight = torch.tensor(
    [float(pos_weight_value)],
    dtype=torch.float32,
    device=DEVICE,
)


criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)


print("pos_weight:", pos_weight.item())

pos_weight: 19.667842864990234


In [58]:
LR_SOURCE = 1e-4
LR_TARGET = 1e-3
LR_CLASSIFIER = 1e-4

WEIGHT_DECAY = 1e-4


optimizer = torch.optim.AdamW(
    [
        {
            "params": source_encoder.parameters(),
            "lr": LR_SOURCE,
        },
        {
            "params": target_encoder.parameters(),
            "lr": LR_TARGET,
        },
        {
            "params": classifier.parameters(),
            "lr": LR_CLASSIFIER,
        },
    ],
    weight_decay=WEIGHT_DECAY,
)

In [59]:
@torch.no_grad()
def estimate_latent_mmd(
    source_encoder,
    target_encoder,
    source_loader,
    target_loader,
    device,
    max_batches=20,
):

    source_encoder.eval()
    target_encoder.eval()

    values = []

    source_iter = iter(source_loader)

    target_iter = iter(target_loader)

    n_batches = min(
        len(source_loader),
        len(target_loader),
        max_batches,
    )

    for _ in range(n_batches):

        xs, _ = next(source_iter)

        (xt,) = next(target_iter)

        xs = xs.to(
            device,
            dtype=torch.float32,
        )

        xt = xt.to(
            device,
            dtype=torch.float32,
        )

        zs = source_encoder(xs)

        zt = target_encoder(xt)

        value = mmd_rbf(
            zs,
            zt,
        )

        values.append(value.item())

    return float(np.mean(values))

In [60]:
initial_mmd = estimate_latent_mmd(
    source_encoder,
    target_encoder,
    source_mmd_loader,
    target_mmd_loader,
    DEVICE,
    max_batches=20,
)

print("Initial latent MMD²:", initial_mmd)

Initial latent MMD²: 0.9688885331153869


In [61]:
LAMBDA_MMD = 0.01

MAX_EPOCHS = 30
PATIENCE = 5


print("lambda MMD:", LAMBDA_MMD)

lambda MMD: 0.01


In [62]:
def train_hda_epoch(
    source_encoder,
    target_encoder,
    classifier,
    source_loader,
    target_loader,
    criterion,
    optimizer,
    lambda_mmd,
    device,
):

    source_encoder.train()
    target_encoder.train()
    classifier.train()

    total_loss = 0.0
    total_cls = 0.0
    total_mmd = 0.0

    n_steps = max(
        len(source_loader),
        len(target_loader),
    )

    source_iter = iter(source_loader)

    target_iter = iter(target_loader)

    for _ in range(n_steps):

        # ------------------------------------
        # Source batch
        # ------------------------------------

        try:
            xs, ys = next(source_iter)

        except StopIteration:

            source_iter = iter(source_loader)

            xs, ys = next(source_iter)

        # ------------------------------------
        # Target batch
        # ------------------------------------

        try:
            (xt,) = next(target_iter)

        except StopIteration:

            target_iter = iter(target_loader)

            (xt,) = next(target_iter)

        # ------------------------------------
        # Device
        # ------------------------------------

        xs = xs.to(
            device,
            dtype=torch.float32,
        )

        ys = ys.to(
            device,
            dtype=torch.float32,
        )

        xt = xt.to(
            device,
            dtype=torch.float32,
        )

        optimizer.zero_grad(set_to_none=True)

        # ------------------------------------
        # Forward
        # ------------------------------------

        zs = source_encoder(xs)

        zt = target_encoder(xt)

        logits_s = classifier(zs)

        # ------------------------------------
        # Losses
        # ------------------------------------

        loss_cls = criterion(
            logits_s,
            ys,
        )

        loss_mmd = mmd_rbf(
            zs,
            zt,
        )

        loss = loss_cls + lambda_mmd * loss_mmd

        # ------------------------------------
        # Backpropagation
        # ------------------------------------

        loss.backward()

        optimizer.step()

        # ------------------------------------
        # Logging
        # ------------------------------------

        total_loss += loss.item()
        total_cls += loss_cls.item()
        total_mmd += loss_mmd.item()

    return {
        "loss": total_loss / n_steps,
        "classification": total_cls / n_steps,
        "mmd": total_mmd / n_steps,
    }

In [63]:
@torch.no_grad()
def predict_domain(
    encoder,
    classifier,
    loader,
    device,
):

    encoder.eval()
    classifier.eval()

    all_probs = []
    all_labels = []

    for x, y in loader:

        x = x.to(
            device,
            dtype=torch.float32,
        )

        z = encoder(x)

        logits = classifier(z)

        probs = torch.sigmoid(logits)

        all_probs.append(probs.cpu().numpy())

        all_labels.append(y.numpy())

    y_true = np.concatenate(all_labels)

    y_prob = np.concatenate(all_probs)

    return y_true, y_prob

In [64]:
def classification_metrics(
    y_true,
    y_prob,
    threshold=0.5,
):

    y_pred = (y_prob >= threshold).astype(int)

    return {
        "pr_auc": average_precision_score(
            y_true,
            y_prob,
        ),
        "roc_auc": roc_auc_score(
            y_true,
            y_prob,
        ),
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "confusion_matrix": confusion_matrix(
            y_true,
            y_pred,
        ),
    }

In [65]:
best_val_pr_auc = -np.inf

best_state = None

epochs_without_improvement = 0

history = []


for epoch in range(
    1,
    MAX_EPOCHS + 1,
):

    train_metrics = train_hda_epoch(
        source_encoder,
        target_encoder,
        classifier,
        source_train_loader,
        target_train_loader,
        criterion,
        optimizer,
        LAMBDA_MMD,
        DEVICE,
    )

    y_val_true, y_val_prob = predict_domain(
        source_encoder,
        classifier,
        source_val_loader,
        DEVICE,
    )

    val_pr_auc = average_precision_score(
        y_val_true,
        y_val_prob,
    )

    val_roc_auc = roc_auc_score(
        y_val_true,
        y_val_prob,
    )

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "classification_loss": train_metrics["classification"],
            "mmd_loss": train_metrics["mmd"],
            "val_pr_auc": val_pr_auc,
            "val_roc_auc": val_roc_auc,
        }
    )

    print(
        f"Epoch {epoch:02d} | "
        f"Loss {train_metrics['loss']:.4f} | "
        f"Cls {train_metrics['classification']:.4f} | "
        f"MMD {train_metrics['mmd']:.4f} | "
        f"Val PR-AUC {val_pr_auc:.4f} | "
        f"Val ROC-AUC {val_roc_auc:.4f}"
    )

    # -------------------------------------------
    # model selection:
    # SOURCE VALIDATION ONLY
    # -------------------------------------------

    if val_pr_auc > best_val_pr_auc:

        best_val_pr_auc = val_pr_auc

        epochs_without_improvement = 0

        best_state = {
            "epoch": epoch,
            "source_encoder": copy.deepcopy(source_encoder.state_dict()),
            "target_encoder": copy.deepcopy(target_encoder.state_dict()),
            "classifier": copy.deepcopy(classifier.state_dict()),
        }

    else:

        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:

        print(
            "Early stopping at epoch",
            epoch,
        )

        break

Epoch 01 | Loss 0.0519 | Cls 0.0518 | MMD 0.0123 | Val PR-AUC 0.9805 | Val ROC-AUC 0.9990
Epoch 02 | Loss 0.0507 | Cls 0.0507 | MMD 0.0011 | Val PR-AUC 0.9806 | Val ROC-AUC 0.9990
Epoch 03 | Loss 0.0506 | Cls 0.0506 | MMD 0.0010 | Val PR-AUC 0.9807 | Val ROC-AUC 0.9990
Epoch 04 | Loss 0.0505 | Cls 0.0505 | MMD 0.0009 | Val PR-AUC 0.9809 | Val ROC-AUC 0.9990
Epoch 05 | Loss 0.0506 | Cls 0.0506 | MMD 0.0008 | Val PR-AUC 0.9809 | Val ROC-AUC 0.9990
Epoch 06 | Loss 0.0503 | Cls 0.0503 | MMD 0.0009 | Val PR-AUC 0.9810 | Val ROC-AUC 0.9990
Epoch 07 | Loss 0.0510 | Cls 0.0510 | MMD 0.0007 | Val PR-AUC 0.9809 | Val ROC-AUC 0.9990
Epoch 08 | Loss 0.0504 | Cls 0.0504 | MMD 0.0006 | Val PR-AUC 0.9810 | Val ROC-AUC 0.9990
Epoch 09 | Loss 0.0500 | Cls 0.0500 | MMD 0.0006 | Val PR-AUC 0.9811 | Val ROC-AUC 0.9990
Epoch 10 | Loss 0.0500 | Cls 0.0500 | MMD 0.0006 | Val PR-AUC 0.9812 | Val ROC-AUC 0.9990
Epoch 11 | Loss 0.0498 | Cls 0.0498 | MMD 0.0005 | Val PR-AUC 0.9812 | Val ROC-AUC 0.9990
Epoch 12 |

In [66]:
history_df = pd.DataFrame(history)

history_df

,epoch,train_loss,classification_loss,mmd_loss,val_pr_auc,val_roc_auc
0,1,0.051939,0.051815,0.012328,0.980473,0.998964
1,2,0.050735,0.050724,0.001104,0.980572,0.998970
2,3,0.050607,0.050597,0.000991,0.980669,0.998974
3,4,0.050462,0.050453,0.000920,0.980891,0.998985
4,5,0.050607,0.050599,0.000835,0.980872,0.998984
5,6,0.050327,0.050319,0.000856,0.980958,0.998988
6,7,0.050991,0.050983,0.000722,0.980897,0.998988
7,8,0.050388,0.050382,0.000627,0.981043,0.998996
8,9,0.049970,0.049964,0.000612,0.981103,0.998999
9,10,0.050024,0.050018,0.000566,0.981231,0.999003


In [67]:
history_df[
    [
        "epoch",
        "classification_loss",
        "mmd_loss",
        "val_pr_auc",
    ]
]

,epoch,classification_loss,mmd_loss,val_pr_auc
0,1,0.051815,0.012328,0.980473
1,2,0.050724,0.001104,0.980572
2,3,0.050597,0.000991,0.980669
3,4,0.050453,0.000920,0.980891
4,5,0.050599,0.000835,0.980872
5,6,0.050319,0.000856,0.980958
6,7,0.050983,0.000722,0.980897
7,8,0.050382,0.000627,0.981043
8,9,0.049964,0.000612,0.981103
9,10,0.050018,0.000566,0.981231


In [68]:
assert best_state is not None


source_encoder.load_state_dict(best_state["source_encoder"])

target_encoder.load_state_dict(best_state["target_encoder"])

classifier.load_state_dict(best_state["classifier"])


print("Best epoch:", best_state["epoch"])

print("Best source val PR-AUC:", best_val_pr_auc)

Best epoch: 30
Best source val PR-AUC: 0.9817705599543884


In [69]:
y_val_true, y_val_prob = predict_domain(
    source_encoder,
    classifier,
    source_val_loader,
    DEVICE,
)


precision_vals, recall_vals, thresholds = precision_recall_curve(
    y_val_true,
    y_val_prob,
)


f1_vals = (
    2
    * precision_vals[:-1]
    * recall_vals[:-1]
    / (precision_vals[:-1] + recall_vals[:-1] + 1e-12)
)


best_idx = np.argmax(f1_vals)


DECISION_THRESHOLD = float(thresholds[best_idx])


print("Decision threshold:", DECISION_THRESHOLD)

print("Source val F1:", f1_vals[best_idx])

Decision threshold: 0.9490832686424255
Source val F1: 0.917947694150029


In [70]:
y_source_true, y_source_prob = predict_domain(
    source_encoder,
    classifier,
    source_test_loader,
    DEVICE,
)


source_metrics = classification_metrics(
    y_source_true,
    y_source_prob,
    threshold=DECISION_THRESHOLD,
)


print("=" * 60)
print("SOURCE TEST")
print("=" * 60)


for key, value in source_metrics.items():

    print(
        key,
        ":",
        value,
    )

SOURCE TEST
pr_auc : 0.9806888965269993
roc_auc : 0.9989621441241106
precision : 0.9076555658750166
recall : 0.9154288772915831
f1 : 0.9115256495669554
confusion_matrix : [[292573   1392]
 [  1264  13682]]


In [71]:
y_target_true, y_target_prob = predict_domain(
    target_encoder,
    classifier,
    target_test_loader,
    DEVICE,
)


target_metrics = classification_metrics(
    y_target_true,
    y_target_prob,
    threshold=DECISION_THRESHOLD,
)


print("=" * 60)
print("TARGET TEST")
print("=" * 60)


for key, value in target_metrics.items():

    print(
        key,
        ":",
        value,
    )

TARGET TEST
pr_auc : 0.15048002113771167
roc_auc : 0.39140789015365074
precision : 0.005476306143178907
recall : 0.001115880600775717
f1 : 0.0018539838224960006
confusion_matrix : [[431349  22519]
 [110999    124]]


In [72]:
target_prevalence = y_target_true.mean()

print("Target attack prevalence:", target_prevalence)

print("Target PR-AUC:", target_metrics["pr_auc"])

Target attack prevalence: 0.19668101
Target PR-AUC: 0.15048002113771167


In [73]:
final_mmd = estimate_latent_mmd(
    source_encoder,
    target_encoder,
    source_mmd_loader,
    target_mmd_loader,
    DEVICE,
    max_batches=20,
)

print("Final latent MMD²:", final_mmd)

Final latent MMD²: 0.0034042119979858397


In [74]:
# ============================================================
# BEFORE / AFTER MMD COMPARISON
# ============================================================

if abs(initial_mmd) > 1e-12:

    mmd_reduction_pct = (initial_mmd - final_mmd) / abs(initial_mmd) * 100.0

else:

    mmd_reduction_pct = np.nan


print("=" * 60)
print("LATENT MMD COMPARISON")
print("=" * 60)

print("Initial MMD²:", initial_mmd)

print("Final MMD²:", final_mmd)

print("MMD reduction (%):", mmd_reduction_pct)

LATENT MMD COMPARISON
Initial MMD²: 0.9688885331153869
Final MMD²: 0.0034042119979858397
MMD reduction (%): 99.64864771522893


In [75]:
HDA_CHECKPOINT_PATH = MODEL_DIR / "hda_mmd.pt"


torch.save(
    {
        "seed": SEED,
        "source_dim": SOURCE_DIM,
        "target_dim": TARGET_DIM,
        "latent_dim": LATENT_DIM,
        "lambda_mmd": LAMBDA_MMD,
        "best_epoch": best_state["epoch"],
        "decision_threshold": DECISION_THRESHOLD,
        "source_encoder_state_dict": source_encoder.state_dict(),
        "target_encoder_state_dict": target_encoder.state_dict(),
        "classifier_state_dict": classifier.state_dict(),
        "source_test_metrics": source_metrics,
        "target_test_metrics": target_metrics,
        "target_prevalence": float(target_prevalence),
        "initial_latent_mmd": initial_mmd,
        "final_latent_mmd": final_mmd,
        "mmd_reduction_pct": float(mmd_reduction_pct),
        "pretrained_source_test_metrics": pretrained_source_metrics,
    },
    HDA_CHECKPOINT_PATH,
)


print("Saved:", HDA_CHECKPOINT_PATH)

Saved: /Users/thonph/Desktop/KLTN/models/hda_mmd.pt


In [76]:
assert SOURCE_DIM == 203
assert TARGET_DIM == 69
assert LATENT_DIM == 64

assert np.isfinite(
    initial_mmd
)

assert np.isfinite(
    final_mmd
)

assert np.isfinite(
    source_metrics["pr_auc"]
)

assert np.isfinite(
    target_metrics["pr_auc"]
)

assert np.isfinite(
    target_prevalence
)

assert best_state is not None


print("=" * 65)
print("HDA + MMD TRAINING CHECK PASSED")
print("=" * 65)

print(
    "Best epoch:",
    best_state["epoch"]
)

print(
    "Source PR-AUC:",
    source_metrics["pr_auc"]
)

print(
    "Target PR-AUC:",
    target_metrics["pr_auc"]
)

print(
    "Target ROC-AUC:",
    target_metrics["roc_auc"]
)

print(
    "Target prevalence:",
    target_prevalence
)

print(
    "Initial latent MMD²:",
    initial_mmd
)

print(
    "Final latent MMD²:",
    final_mmd
)

print(
    "MMD reduction (%):",
    mmd_reduction_pct
)

HDA + MMD TRAINING CHECK PASSED
Best epoch: 30
Source PR-AUC: 0.9806888965269993
Target PR-AUC: 0.15048002113771167
Target ROC-AUC: 0.39140789015365074
Target prevalence: 0.19668101
Initial latent MMD²: 0.9688885331153869
Final latent MMD²: 0.0034042119979858397
MMD reduction (%): 99.64864771522893
